# 00 — Environment and CUDA check

**Purpose:** Verify the active Conda kernel, Windows resources, PyTorch/CUDA, packages, disk and writable outputs.

**Required inputs:** `config.local.yaml`; installed Python environment. No dataset is needed.

**Saved outputs:** `outputs/environment_report.json`, `outputs/environment_packages.txt`.

## 1. Load the central configuration

The root is found from the notebook location or the installed package. Set `LEUKEMIA_YOLO_CONFIG` only for synthetic smoke tests or an alternate config.

In [ ]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "config.py").is_file():
        sys.path.insert(0, str(candidate))
        break
from src.config import load_config, ensure_outputs
cfg = load_config()
out = ensure_outputs(cfg)
os.chdir(cfg["root"])
print("Project root:", cfg["root"])
print("Configuration:", cfg["config_path"])

## 2. Inspect environment

PASS/WARNING/FAIL are colored. System RAM and NVIDIA dedicated VRAM are separate fields.

In [ ]:
from src.environment import inspect_environment, status_table
report = inspect_environment(cfg)
table = status_table(report)
display(table[["check", "value", "status", "detail"]])

## 3. Verify GPU gate

A CPU-only wheel or unavailable driver produces an explicit failure. Training also checks this gate in its child process.

In [ ]:
from src.environment import require_cuda
try:
    gpu = require_cuda(cfg["device"])
    print("PASS — CUDA GPU:", gpu)
except RuntimeError as exc:
    print("FAIL — CUDA training unavailable:", exc)

## 4. Saved report and status

Reports are generated even when CUDA is absent, so the development laptop can complete this notebook.

In [ ]:
assert (out / "environment_report.json").is_file()
assert (out / "environment_packages.txt").is_file()
print("PASS — environment report saved" if report["cuda_ready"] and report["kernel_ready"] else
      "WARNING — report saved; resolve FAIL/WARNING rows before GPU training")

## Next notebook

Open **01_BCCD_Dataset_Inspection.ipynb** after setting the dataset path in `config.local.yaml`.